# 이전 4시간 구성 · 선택 복습
현재 필수 수업은 [교재 안내](../../README.md)를 따릅니다. 이 파일은 이전 상세 실습 보존본입니다.


# USB 카메라 장면 설명과 시간별 기록

**목표:** 최신 프레임만 선택해 분석하고 원본·시각·결과를 함께 남깁니다.

60분 수업: 그림·개념 설명 → 실행 전 예측 → 코드 실행 → 원문·기록 비교. 앞 세 교시는 휴식 10분 포함, 마지막 교시는 종합 평가 포함입니다.

[교재 안내](../../README.md) · [슬라이드 82~91쪽](https://docs.google.com/presentation/d/1IIy5LG5Yue5sJR5DOzmtW6jOuX0DDIZb3LsswTamjcw/edit) · [배포 PDF](../../../슬라이드/강의자료_2일차_v2.pdf#page=82)

**커널:** Python (Day2 LangChain). 설치는 개인 컨테이너 Terminal에서 `bash /workspace/04.LLM_v2/scripts/install_langchain.sh`를 실행합니다. 설치가 끝나면 커널을 선택하고 아래 환경 셀부터 실행하세요. 본문 코드는 실행 가능한 기준 구현이며 마지막에는 별도의 TODO가 있습니다. 코드 작성 여부와 수정할 파일을 TODO 안내에서 확인하세요. TODO는 수업 마지막 10~15분 동안 진행합니다.

In [ ]:
from pathlib import Path
import json, os, sys
ROOT=next((p for p in (Path.cwd(),*Path.cwd().parents) if (p/'04.LLM_v2').is_dir()),None)
if ROOT is None:raise RuntimeError('배포 폴더 안에서 노트북을 여세요.')
COURSE=ROOT/'05.VLM_v2'
sys.path.insert(0,str(COURSE))
from vlm_v2.core import *
print('Python:',sys.executable)
print('교재:',COURSE)

## 1. 영상은 시간 순서의 이미지입니다

카메라는 초당 여러 프레임을 만듭니다.

VLM 요청은 프레임 생성보다 느릴 수 있습니다.

한 장의 답은 촬영 시점의 관찰입니다.

중간 프레임의 사건은 확인하지 못할 수 있습니다.

30 FPS는 1초에 30장이라는 뜻입니다. 모델이 한 장 처리에 몇 초 걸리면 모든 프레임을 보내는 순간 요청이 쌓입니다. 처리율과 입력률을 숫자로 비교하고 어느 장면을 버릴지 정책을 먼저 정합니다.

**설명해 보기:** 입력·중간값·출력 중 이 단계에서 바뀌는 것은 무엇인가요?

![영상은 시간 순서의 이미지입니다](../../assets/03_camera_records-01.svg)

## 2. 모든 프레임을 큐에 넣지 않습니다

촬영 스레드는 최신 프레임 하나를 갱신합니다.

분석은 한 번에 한 요청만 수행합니다.

요청 시작 간격을 interval로 제한합니다.

완료 후 다음 최신 프레임을 선택합니다.

지연이 interval보다 길면 실제 요청 주기도 길어집니다. interval=10은 정확히 10초마다 반드시 답이 온다는 뜻이 아닙니다. 오래된 대기 프레임을 버려 최신성을 유지하지만 미관찰 구간은 늘어납니다.

**설명해 보기:** 입력·중간값·출력 중 이 단계에서 바뀌는 것은 무엇인가요?

## 3. 최신 프레임 저장소를 실험합니다

새 프레임이 들어오면 이전 슬롯을 대체합니다.

sequence는 촬영 순서 번호입니다.

마지막 처리 번호보다 새로운 것만 가져옵니다.

하나만 저장하는 것과 모든 큐 저장을 비교합니다.

아래는 실제 카메라가 아닌 자료구조 실험입니다. A·B·C를 연속 입력하면 분석자는 마지막 C만 보게 됩니다. 이 결과를 실제 시간·프레임의 의미로 바꾸어 설명하세요.

**설명해 보기:** 입력·중간값·출력 중 이 단계에서 바뀌는 것은 무엇인가요?

In [ ]:
from vlm_v2.sampling import LatestFrame
latest=LatestFrame()
latest.put('frame A',1.0);latest.put('frame B',2.0);latest.put('frame C',3.0)
assert latest.get_after(0)==(3,3.0,'frame C')
assert latest.get_after(3) is None
print('최신 프레임 선택 PASS')

## 4. 원본 프레임과 기록을 연결합니다

선택 프레임을 JPEG로 저장합니다.

ID·촬영 UTC·sequence·SHA-256을 남깁니다.

질문·응답 원문·검사 상태·시간을 남깁니다.

한 요청이 끝날 때마다 JSONL에 추가합니다.

JSONL은 한 줄에 하나의 JSON 객체입니다. 첫 줄은 실행 설정 header이고 다음 줄은 관찰입니다. 중간 요청이 실패해도 앞 기록이 남습니다. monotonic 시간은 간격 계산용, UTC는 기록을 읽는 시각 정보용입니다.

**설명해 보기:** 입력·중간값·출력 중 이 단계에서 바뀌는 것은 무엇인가요?

![원본 프레임과 기록을 연결합니다](../../assets/03_camera_records-04.svg)

## 5. 실행 전 저장 경로를 dry-run으로 확인합니다

dry-run은 모델 요청 없이 프레임을 저장합니다.

dry_run 상태를 실제 관찰과 구분합니다.

입력·출력 경로와 카메라 접근을 확인합니다.

다음 실행은 새 폴더에 실제 결과를 만듭니다.

학생은 개인 컨테이너 Terminal에서 명령을 입력합니다. 카메라는 로컬 담당자 한 명만 열어 두 수강생이 같이 관찰합니다. 연결 문제가 있으면 녹화 영상을 선택해 동일 기록 흐름을 확인할 수 있습니다.

**설명해 보기:** 입력·중간값·출력 중 이 단계에서 바뀌는 것은 무엇인가요?

**개인 컨테이너 Terminal 명령** (Ollama 접속 명령은 Jetson 호스트):

```bash
cd /workspace
PY=Data/venvs/day2-langchain/bin/python
"$PY" 05.VLM_v2/scripts/camera_vlm.py \
  --source camera --max-requests 3 \
  --interval 10 --dry-run
```

## 6. USB 카메라로 실제 장면을 기록합니다

로컬 컨테이너에서 카메라를 한 번 실행합니다.

두 수강생이 관찰·질문·기록 역할을 교대합니다.

--preview 화면은 Jetson 모니터에 표시됩니다.

Ctrl+C 또는 요청 수 도달로 종료합니다.

창은 모델 요청 중에는 즉시 갱신되지 않을 수 있습니다. 촬영 스레드는 최신 프레임을 유지합니다. Ctrl+C 뒤 카메라를 해제하며, 다른 프로그램이 동시에 카메라를 열지 않도록 기존 미리보기를 종료하세요.

**설명해 보기:** 입력·중간값·출력 중 이 단계에서 바뀌는 것은 무엇인가요?

**개인 컨테이너 Terminal 명령** (Ollama 접속 명령은 Jetson 호스트):

```bash
cd /workspace
PY=Data/venvs/day2-langchain/bin/python
"$PY" 05.VLM_v2/scripts/camera_vlm.py \
  --source camera --camera-index 0 \
  --interval 10 --max-requests 3 --preview
```

## 7. 장면을 바꾸고 관찰 시각을 적습니다

첫 구간은 기준 장면을 보여 줍니다.

다음 구간에서 물체 배치나 시야를 바꿉니다.

변경 시각은 사람이 별도로 기록합니다.

VLM의 설명과 실제 변경을 비교합니다.

프레임 사이에 잠깐 나타났다 사라진 물체는 기록에 없을 수 있습니다. “변화 없음”은 실제 변화가 없었다는 확정이 아닙니다. 기준 사진·가림·복원 같은 간단한 상황으로 차이를 관찰하세요.

**설명해 보기:** 입력·중간값·출력 중 이 단계에서 바뀌는 것은 무엇인가요?

## 8. 녹화 영상으로 같은 흐름을 재생합니다

카메라 장애나 복습에는 배포 영상을 사용합니다.

파일 영상도 최신 프레임 정책으로 처리합니다.

원본 영상의 모든 장면을 분석한 것은 아닙니다.

녹화 입력임을 결과와 평가에 표시합니다.

영상의 재생 속도와 모델 지연에 따라 선택 프레임이 달라질 수 있습니다. 동일 프레임 비교가 필요하면 이미지 목록 입력으로 명시적으로 고정하세요. 영상은 선택 복습이지만 카메라 접근이 어려운 학생의 대체 입력이 됩니다.

**설명해 보기:** 입력·중간값·출력 중 이 단계에서 바뀌는 것은 무엇인가요?

**개인 컨테이너 Terminal 명령** (Ollama 접속 명령은 Jetson 호스트):

```bash
cd /workspace
PY=Data/venvs/day2-langchain/bin/python
"$PY" 05.VLM_v2/scripts/camera_vlm.py \
  --source video --interval 10 \
  --max-requests 3
```

## 9. 실제 관찰 JSONL을 읽습니다

터미널에 출력된 로그 경로를 선택합니다.

전체 파일 목록에서 자신의 실행을 구분합니다.

각 행의 시각·상태·원본 경로를 읽습니다.

오류 행을 지우지 않습니다.

아래 셀은 현재 폴더의 실행 목록만 보여 줍니다. 실제 로그를 선택하는 것은 다음 교시에서 명시적으로 합니다. 모의 예제를 실제 카메라 관찰로 혼동하지 않도록 입력 출처를 확인하세요.

**설명해 보기:** 입력·중간값·출력 중 이 단계에서 바뀌는 것은 무엇인가요?

In [ ]:
logs=sorted(OUTPUTS.glob('stream-*/observations.jsonl'))
for path in logs[-5:]:print(path)
print('자신의 실행 경로를 다음 교시 OBSERVATION_LOG에 넣으세요.')

## 10. 3교시 정리 · 시간과 누락을 설명합니다

최신 슬롯과 처리 간격의 역할을 설명합니다.

촬영 시각과 응답 도착 시각을 구분합니다.

원본·ID·해시·상태의 연결을 확인합니다.

다음 교시는 선택한 기록으로 변화 요약을 합니다.

이론 20분, 실행·공동 관찰 25분, 정리 5분, 휴식 10분입니다. interval=3/10 비교는 추가 실험입니다. 요청 수를 늘리기 전에 무엇을 비교할지 적고, 결과가 없는 시간 구간을 따로 표시하세요.

**설명해 보기:** 입력·중간값·출력 중 이 단계에서 바뀌는 것은 무엇인가요?

## 마지막 TODO — VLM 요청 간격 판단

**코드 작성: 있음 · 10분 · 수업 시간 안에 진행**

수정 파일: [practice/request_interval.py](../../practice/request_interval.py). JupyterLab 편집기에서 함수를 완성하고 저장한 뒤 아래 로딩 셀과 검사 셀을 다시 실행하세요. 노트북은 학생 파일을 덮어쓰지 않습니다.

**입력·출력 계약:** now와 last_request는 단조 증가하는 시계의 초 값입니다. interval은 bool이 아닌 양의 유한한 숫자여야 하며 아니면 ValueError입니다. last_request=None이면 첫 요청이므로 True, 그 외에는 now-last_request >= interval일 때만 True를 반환하세요. 경계에서 같을 때도 요청합니다. now/last_request는 유효한 숫자가 제공됩니다.

**힌트:** 첫 요청을 먼저 처리하고, 나머지는 경과 시간과 interval을 비교하세요. >와 >=의 차이를 시험하세요.

미완성이면 마지막 검사에서 NotImplementedError가 나는 것이 정상입니다. 검사 통과 후 아래 적용 예제의 중간값을 설명하세요. 이 예제는 학생 함수를 직접 사용하며 본문의 기준 클래스/카메라 프로그램을 자동 교체하지 않습니다.

In [ ]:
import importlib.util, inspect, os
runner_path=COURSE/'practice/runner.py'
spec=importlib.util.spec_from_file_location('day2_todo_runner',runner_path)
runner=importlib.util.module_from_spec(spec)
spec.loader.exec_module(runner)
VARIANT=os.environ.get('DAY2_TODO_VARIANT','exercise')
task=runner.load('request_interval',VARIANT)
print('선택 구현:',VARIANT)
print(inspect.getsource(task.should_request))

In [ ]:
print(runner.check('request_interval',task))

times=[0,1,2.9,3,4,6,9]
selected=[];last_request=None
for now in times:
    if task.should_request(now,last_request,3):
        selected.append(now)
        last_request=now  # 실제 요청을 선택한 순간에만 갱신
print('요청 시각:',selected)
assert selected==[0,3,6,9]
print('이 셀은 시간 선택 모의 실험이며 카메라를 열거나 모델을 호출하지 않습니다.')

### TODO 완료와 참고 정답

- [ ] 함수를 직접 작성하고 검사 셀을 통과했습니다.
- [ ] 경계 입력 한 개를 추가해 예상 결과를 설명했습니다.
- [ ] 적용 예제의 입력과 반환값을 설명했습니다.

강사가 정한 직접 시도 시간 뒤 [참고 정답](../../practice/solutions/request_interval.py)을 비교하세요. 참고 정답을 실행했다고 학생 파일이 완성되는 것은 아닙니다.

**해설:** 프레임마다 요청하지 않고 최소 시작 간격을 제한합니다. 실제 프로그램에서는 모델 처리가 더 느리면 요청 주기도 더 길어집니다. now는 wall clock이 아니라 monotonic 시계입니다.

추가 모델 요청 없이 함수와 데이터 흐름을 검사합니다. 실제 모델·카메라 결과 평가는 본문 실습에서 수행합니다.

## 완료 확인

실제 입력과 출력 기록을 확인하고, 형식·근거·지연을 구분해 설명합니다. 오류는 숨기지 않고 원문과 함께 남깁니다. 추가 실험은 한 조건만 바꿔 비교합니다.